# ETL Bronze to Silver

## 1. Configuração e Importações

In [9]:
import pandas as pd
import numpy as np 
import re 
import os 
import unicodedata
import psycopg2
from psycopg2.extras import execute_batch

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

In [10]:
INPUT_FILE = 'Data Layer/raw/data_raw.csv'
OUTPUT_DIR = 'Data Layer/silver/data'
OUTPUT_FILE_CSV = 'data_silver.csv'

DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'amazon_sales',
    'user': 'postgres',
    'password': 'postgres'
}

## 2. Funções Auxiliares

### 2.1 Extração do ASIN

In [11]:
def extract_asin(url):
    """
        '/dp/B08N5WRWNW/' → 'B08N5WRWNW'
        '/gp/product/B09G9HD6PD' → 'B09G9HD6PD'
    """
    if pd.isna(url) or url == '':
        return None
    
    url_str = str(url).strip()
    
    # Pattern 1: /dp/ASIN or /gp/product/ASIN
    match = re.search(r'/(?:dp|gp/product|product)/([A-Z0-9]{10})(?:[/?]|$)', url_str, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    
    # Pattern 2: Any 10-char alphanumeric (conservative fallback)
    if 'amazon.' in url_str.lower():
        match = re.search(r'(?:^|[^A-Z0-9])([A-Z0-9]{10})(?:$|[^A-Z0-9])', url_str, re.IGNORECASE)
        if match:
            candidate = match.group(1).upper()
            if re.fullmatch(r'[A-Z0-9]{10}', candidate):
                return candidate
    
    return None

### 2.2 Extração da Marca
**Definimos a lógica para identificar a marca de cada produto analisando seu título.**  
Precisamos filtrar palavras genéricas (stopwords) para capturar apenas o nome real da marca.


In [12]:
# Palavras que devem ser ignoradas (não são marcas)
STOPWORDS = {'the', 'a', 'an', 'new', 'latest', '2025', '2024', 'portable', 
             'wireless', 'with', 'for', 'and', 'by', 'from', 'brand', 'official'}

def extract_brand(title):
    """
    Extrai a marca do título do produto.  
    Exemplos:
        'Samsung Galaxy S21 - 128GB' → 'samsung'
    """
    if pd.isna(title) or title == '':
        return 'unknown'
    
    # Normaliza o texto (remove acentos, caracteres especiais)
    title_clean = unicodedata.normalize('NFKC', str(title)).strip()
    
    # Pega a primeira parte do título (antes de -, |, :, etc)
    segment = re.split(r'[-–|:()\\[,/]', title_clean, maxsplit=1)[0]
    
    # Caso especial: "Marca: XYZ"
    brand_match = re.search(r'^\\s*(?:brand|manufacturer)\\s*[:\\-]\\s*([A-Za-z0-9\\-\\+\\. ]{2,})', 
                            segment, flags=re.IGNORECASE)
    if brand_match:
        segment = brand_match.group(1)
    
    # Extrai palavras alfanuméricas
    tokens = re.findall(r'[A-Za-z0-9\\+\\.\\-]+', segment)
    
    if not tokens:
        return 'unknown'
    
    # Pega a primeira palavra que não seja stopword
    candidate = tokens[0].lower()
    if candidate in STOPWORDS or len(candidate) <= 1:
        candidate = tokens[1].lower() if len(tokens) > 1 else 'unknown'
    
    return candidate

### 2.3 Inferência de Categoria
**Construímos uma função que classifica produtos em categorias usando palavras-chave.**  
Isso nos permitirá segmentar análises por tipo de produto (Laptop, Audio, Mobile, etc.).


In [13]:
def infer_category(title):
    """
    Infer product category from title keywords.
    """
    if pd.isna(title) or str(title).strip() == '':
        return 'Other'

    title_norm = unicodedata.normalize('NFKC', str(title)).lower()
    title_norm = re.sub(r"[^a-z0-9\s]", " ", title_norm)
    title_norm = re.sub(r"\s+", " ", title_norm).strip()

    if title_norm == "":
        return 'Other'

    category_rules = [
        # Core electronics
        (r'\b(laptops?|notebooks?|macbooks?|chromebooks?|ultrabooks?|gaming\s*laptops?|surface\s*laptops?)\b', 'Laptop'),
        (r'\b(headphones?|headsets?|earbuds?|earphones?|tws|ear\s*buds?|neckbands?|speakers?|soundbars?|subwoofers?|woofers?|microphones?|mics?|lavalier|airpods?|earpods?|buds)\b', 'Audio'),
        (r'\b(cameras?|dslr|mirrorless|webcams?|action\s*cams?|gopro|camcorders?|instax|polaroid|security\s*cameras?|dash\s*cams?|picture\s*frames?)\b', 'Camera'),
        (r'\b(phones?|iphone\s?\d*|android|smartphones?|oneplus|samsung\s*galaxy|mobiles?|pixel|motorola|nokia|xiaomi|redmi|oppo|vivo|realme|infinix|tecno)\b', 'Mobile'),
        (r'\b(tablets?|ipads?|tab\b|galaxy\s*tabs?|surface\s*pros?|kindles?|fire\s*tablets?|fire\s*hd|ereaders?)\b', 'Tablet'),
        (r'\b(ssd|solid\s*state\s*drives?|hard\s*drives?|hdd|micro\s*sd|sd\s*cards?|flash\s*drives?|pen\s*drives?|pendrives?|memory\s*cards?|thumb\s*drives?|external\s*drives?|portable\s*drives?|nas|nvme|ram|memory\s*modules?)\b', 'Storage'),
        (r'\b(smartwatches?|smart\s*watches?|fitness\s*bands?|smart\s*bands?|fitbands?|wearables?|fitbit|garmin|amazfit)\b', 'Wearable'),
        (r'\b(router|routers|modems?|mesh|wifi\s*(?:system|router|kit)|range\s*extenders?|extenders?|repeater|boosters?|ethernet\s*(?:switch|adapter|hub)|access\s*point|powerline)\b', 'Networking'),
        (r'\b(processors?|cpus?|ryzen|intel\s*(?:core|pentium|celeron)|motherboards?|mainboards?|coolers?|heatsinks?|power\s*supplies?|psus?|graphics\s*cards?|gpus?|gpu)\b', 'Components'),
        (r'\b(mice|mouse|keyboards?|monitors?|adapters?|cables?|chargers?|hubs?|docks?|power\s*banks?|usb|stylus|cases?|covers?|stands?|mounts?|tripods?|gimbals?|screen\s*protectors?|cleaning\s*kits?|cooling\s*pads?|webcams?)\b', 'Accessory'),
        (r'\b(tvs?|televisions?|projectors?|smart\s*tvs?|oled|qled|uhd|4k\s*tvs?|8k\s*tvs?)\b', 'TV/Display'),
        (r'\b(playstation|xbox|ps\d|controllers?|gaming|consoles?|nintendo|switch|steam\s*deck|joysticks?|vr\s*headsets?|oculus|meta\s*quest)\b', 'Gaming'),
        # Peripherals & printing
        (r'\b(printers?|scanners?|inkjet|laserjet|label\s*makers?|plotters?)\b', 'Printing'),
        (r'\b(ink|toners?|cartridges?|drums?|refills?)\b', 'Printing Supplies'),
        # Power & smart home
        (r'\b(batteries?|power\s*banks?|alkaline|chargers?|charging\s*stations?|power\s*stations?|surge\s*protectors?|ups)\b', 'Power'),
        (r'\b(smart\s*plugs?|smart\s*bulbs?|smart\s*lights?|smart\s*locks?|doorbells?|alexa|echo|ring\s*cameras?|smart\s*displays?|homekit|smart\s*thermostats?|smart\s*home)\b', 'Smart Home'),
        # Appliances & office
        (r'\b(vacuums?|air\s*fryers?|blenders?|coffee\s*makers?|microwaves?|refrigerators?|fridges?|dishwashers?|washers?|dryers?|humidifiers?|purifiers?)\b', 'Home Appliance'),
        (r'\b(drones?|quad\s*copters?|fpv\s*drones?|mini\s*drones?)\b', 'Drone'),
        (r'\b(calculators?|ti\s*\d+|graphing\s*calculator)\b', 'Calculator/Office'),
        (r'\b(laminators?|laminating|tapes?|pencils?|pens?|markers?|notebooks?|stationery|paper|envelopes?|folders?|binders?|highlighters?|staplers?|labels?)\b', 'Office Supplies'),
        (r'\b(chairs?|desks?|standing\s*desks?|office\s*chairs?)\b', 'Office Furniture'),
        (r'\b(pet\s*pads?|litter|dog\s*pads?)\b', 'Pet Supplies'),
    ]

    for pattern, category in category_rules:
        if re.search(pattern, title_norm):
            return category

    return 'Other'

### 2.4 Parsing do Target (Unidades Vendidas)
**Desenvolvemos uma função robusta para converter textos de vendas em números.**  
Precisamos lidar com diversos formatos: "6K+", "menos de 100", "novo no mercado", etc.


In [14]:
def parse_units_sold(text):
    """
    Parse 'bought_in_last_month' column robustly.
    
    Cases:
        '6K+ bought in past month' → 6000.0
        '1.5k bought' → 1500.0
        '300+ bought' → 300.0
        'Less than 100 bought' → 50.0 (midpoint heuristic)
        'New to market' / 'Just launched' → 0.0
        Invalid → NaN
    """
    if pd.isna(text) or text == '':
        return np.nan
    
    text_clean = unicodedata.normalize('NFKC', str(text)).strip().lower()
    
    # Case 1: New products (zero sales)
    zero_phrases = ['new to market', 'just launched', 'be the first', 
                    'no sales yet', 'recently added']
    if any(phrase in text_clean for phrase in zero_phrases):
        return 0.0
    
    # Case 2: "Less than X"
    match_less = re.search(r'less\s+than\s+(\d+(?:\.\d+)?)\s*([km]?)', text_clean)
    if match_less:
        base_number = float(match_less.group(1))
        suffix = match_less.group(2)
        
        if suffix == 'k':
            base_number *= 1000
        elif suffix == 'm':
            base_number *= 1_000_000
        
        return base_number * 0.5  # Midpoint heuristic
    
    # Case 3: "6K+" or "1.5k" or "300+"
    match_num = re.search(r'(\d+(?:\.\d+)?)\s*([km]?)\s*\+?', text_clean)
    if match_num:
        base_number = float(match_num.group(1))
        suffix = match_num.group(2)
        
        if suffix == 'k':
            return base_number * 1000
        elif suffix == 'm':
            return base_number * 1_000_000
        else:
            return base_number
    
    return np.nan

### 2.5 Conversão de Preço & Extração de Cupom

**Criamos funções para limpar valores monetários e extrair descontos de cupons.**  
Precisamos transformar strings como "$1,299.99" em números puros para análises posteriores.

In [15]:
def convert_to_float(value):
    """Convert price strings to float."""
    if pd.isna(value) or value == '':
        return np.nan
    
    value = str(value).strip()
    match = re.search(r'(\\d+(?:,\\d{3})*(?:\\.\\d+)?)', value)
    
    if not match:
        return np.nan
    
    number_str = match.group(1).replace(',', '')
    return float(number_str)


def extract_coupon_percentage(coupon_text):
    """Extract coupon discount percentage."""
    if pd.isna(coupon_text) or coupon_text == '' or 'No Coupon' in str(coupon_text):
        return 0.0
    
    match = re.search(r'(\\d+(?:\\.\\d+)?)%', str(coupon_text))
    if match:
        return float(match.group(1))
    return 0.0

## 3. Carregando os Dados Brutos
**Iniciamos o processo ETL carregando o arquivo CSV com dados não processados.**  
Verificamos o tamanho do dataset para entender o volume de trabalho pela frente.


In [16]:
print("="*80)
print("ETL BRONZE -> SILVER")
print("="*80)

print("\nCarregando dados Bronze...")
df = pd.read_csv(INPUT_FILE)
print(f"   Carregado: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print(f"   Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

rows_initial = len(df)


ETL BRONZE -> SILVER

Carregando dados Bronze...
   Carregado: 42,675 linhas x 16 colunas
   Memoria: 71.84 MB


## 4. Extraindo ASIN e Removendo Duplicatas
**Aqui começamos a limpar os dados extraindo o código ASIN de cada produto.**  
Precisamos fazer isso ANTES de qualquer filtragem pois produtos duplicados podem ter informações diferentes em cada coleta.


In [17]:
print("\nExtraindo ASIN (codigo unico do produto)...")

df['asin'] = df['product_url'].apply(extract_asin)

with_asin = df['asin'].notna().sum()
without_asin = df['asin'].isna().sum()

print(f"   Com ASIN: {with_asin:,} ({with_asin/len(df)*100:.1f}%)")
print(f"   Sem ASIN: {without_asin:,} ({without_asin/len(df)*100:.1f}%)")

if with_asin > 0:
    unique_asins = df['asin'].nunique()
    duplicates = with_asin - unique_asins
    print(f"   ASINs unicos: {unique_asins:,}")
    print(f"   Duplicatas detectadas: {duplicates:,}")


Extraindo ASIN (codigo unico do produto)...
   Com ASIN: 35,114 (82.3%)
   Sem ASIN: 7,561 (17.7%)
   ASINs unicos: 8,377
   Duplicatas detectadas: 26,737


In [18]:
print("\nRemovendo duplicatas por ASIN...")

df_with_asin = df[df['asin'].notna()].copy()
df_without_asin = df[df['asin'].isna()].copy()

if len(df_with_asin) > 0:
    df_with_asin['collected_at'] = pd.to_datetime(df_with_asin['collected_at'], errors='coerce')
    df_with_asin = df_with_asin.sort_values(['asin', 'collected_at'], ascending=[True, False])
    df_dedup = df_with_asin.drop_duplicates(subset=['asin'], keep='first')
    df = pd.concat([df_dedup, df_without_asin], axis=0, ignore_index=True)
    
    rows_removed = rows_initial - len(df)
    print(f"   Antes: {rows_initial:,}")
    print(f"   Depois: {len(df):,}")
    print(f"   Duplicatas removidas: {rows_removed:,} ({rows_removed/rows_initial*100:.1f}%)")


Removendo duplicatas por ASIN...
   Antes: 42,675
   Depois: 15,938
   Duplicatas removidas: 26,737 (62.7%)


## 5. Convertendo Tipos de Dados
**Agora convertemos cada coluna para seu tipo adequado.**  
Precisamos fazer isso para que operações matemáticas funcionem corretamente e para criar features temporais úteis.


In [19]:
print("\nConvertendo tipos de dados...")

df['collected_at'] = pd.to_datetime(df['collected_at'], errors='coerce')

df['date'] = df['collected_at'].dt.normalize()
df['time'] = df['collected_at'].dt.time
df['hour'] = df['collected_at'].dt.hour
df['day_of_week'] = df['collected_at'].dt.dayofweek
df['day_name'] = df['collected_at'].dt.day_name()

print(f"   Periodo dos dados: {df['date'].min()} ate {df['date'].max()}")


Convertendo tipos de dados...
   Periodo dos dados: 2025-08-21 00:00:00 ate 2025-08-30 00:00:00


In [20]:
# Limpeza de avaliacoes e reviews
df['rating'] = pd.to_numeric(df['rating'].str.extract(r'([\d\.]+)', expand=False), errors='coerce').clip(0, 5)
df['number_of_reviews'] = df['number_of_reviews'].str.replace(',', '', regex=False).fillna('0').astype(int)

print("   Avaliacoes e reviews convertidos")

   Avaliacoes e reviews convertidos


In [21]:
# Conversao da variavel alvo
print("   Convertendo unidades vendidas (target)...")

df['units_sold_last_month'] = df['bought_in_last_month'].apply(parse_units_sold)

parsed_count = df['units_sold_last_month'].notna().sum()
print(f"      {parsed_count:,} registros com target valido ({parsed_count/len(df)*100:.1f}%)")

   Convertendo unidades vendidas (target)...
      11,585 registros com target valido (72.7%)


In [22]:
# Conversao de precos
df['current/discounted_price'] = df['current/discounted_price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).apply(lambda x: float(x) if x and x != '0' else np.nan)
df['listed_price'] = df['listed_price'].apply(convert_to_float)
df['price_on_variant'] = df['price_on_variant'].apply(convert_to_float)

print("   Precos convertidos para numeros")

   Precos convertidos para numeros


In [23]:
# Conversao de flags para booleanos
df['is_best_seller'] = df['is_best_seller'].apply(lambda x: True if str(x).strip() == 'Best Seller' else False).astype(bool)
df['is_sponsored'] = df['is_sponsored'].apply(lambda x: True if str(x).strip() == 'Sponsored' else False).astype(bool)
df['buy_box_availability'] = df['buy_box_availability'].apply(lambda x: True if str(x).strip() == 'Add to cart' else False).astype(bool)

df['title'] = df['title'].astype('string')
df['time'] = df['time'].astype('string')

print("   Conversao de tipos concluida")

   Conversao de tipos concluida


## 6. Criando Novas Features (Engenharia de Atributos)

**Passamos agora para a criação de colunas derivadas que agregarão valor às análises.**  
Vamos extrair informações escondidas nos dados brutos e calcular métricas importantes.

### 6.1 Extração de Marca
**Aplicamos a função de extração de marca em todos os produtos.**  
Isso nos dá visibilidade de quantas marcas diferentes existem no dataset.


In [24]:
df['brand'] = df['title'].apply(extract_brand)
print(f"      {df['brand'].nunique():,} marcas unicas")

      628 marcas unicas


### 6.2 Classificação por Categoria
**Classificamos cada produto em categorias usando a função de inferência.**  
Verificamos a distribuição para entender quais tipos de produtos são mais comuns.


In [25]:
print("   Inferindo categorias...")
df['category'] = df['title'].apply(infer_category)
category_dist = df['category'].value_counts(dropna=False)
print(f"      {len(category_dist)} categorias identificadas")
for cat, count in category_dist.head(10).items():
    print(f"         - {cat}: {count:,} ({count/len(df):.1%})")

other_pct = (category_dist.get('Other', 0) / len(df)) * 100

category_summary = (
    category_dist.to_frame(name='count')
    .assign(share=lambda s: (s['count'] / len(df) * 100).round(2))
)

   Inferindo categorias...
      22 categorias identificadas
         - Audio: 2,529 (15.9%)
         - Other: 2,294 (14.4%)
         - Camera: 2,174 (13.6%)
         - Power: 1,930 (12.1%)
         - Accessory: 1,593 (10.0%)
         - Laptop: 1,251 (7.8%)
         - Mobile: 1,198 (7.5%)
         - Printing: 609 (3.8%)
         - Storage: 526 (3.3%)
         - Networking: 335 (2.1%)


### 6.3 Unificação de Preços
**Criamos uma coluna única de preço final usando lógica em cascata.**  
Tentamos: 1º preço com desconto → 2º preço da variante → 3º preço original. Assim maximizamos a cobertura.


In [26]:
print("   Construindo preco final (logica waterfall)...")
df['final_price'] = df['current/discounted_price'].fillna(df['price_on_variant']).fillna(df['listed_price'])

df['price_source'] = np.select(
    [
        df['current/discounted_price'].notna(),
        df['current/discounted_price'].isna() & df['price_on_variant'].notna(),
        df['current/discounted_price'].isna() & df['price_on_variant'].isna() & df['listed_price'].notna()
    ],
    ['discounted', 'variant', 'original'],
    default='none'
)

missing_price = df['final_price'].isna().sum()
print(f"      Precos faltantes: {missing_price:,} ({missing_price/len(df)*100:.1f}%)")

   Construindo preco final (logica waterfall)...
      Precos faltantes: 3,031 (19.0%)


### 6.4 Preenchimento Inteligente de Preços Faltantes

**Usamos estatísticas inteligentes para preencher preços que ainda estão vazios.**  
Priorizamos: 1º mediana da mesma marca+categoria → 2º mediana da marca → 3º mediana da categoria → 4º mediana geral.

In [27]:
if missing_price > 0:
    print("   Aplicando imputacao de precos...")
    
    median_brand_cat = df.groupby(['brand', 'category'])['final_price'].median()
    median_brand = df.groupby('brand')['final_price'].median()
    median_cat = df.groupby('category')['final_price'].median()
    global_median = df['final_price'].median()
    
    def impute_price(row):
        if pd.notna(row['final_price']):
            return row['final_price'], 'original'
        
        key = (row['brand'], row['category'])
        if key in median_brand_cat and pd.notna(median_brand_cat[key]):
            return median_brand_cat[key], 'imputed_brand_cat'
        
        if row['brand'] in median_brand and pd.notna(median_brand[row['brand']]):
            return median_brand[row['brand']], 'imputed_brand'
        
        if row['category'] in median_cat and pd.notna(median_cat[row['category']]):
            return median_cat[row['category']], 'imputed_category'
        
        return global_median, 'imputed_global'
    
    imputed_data = df.apply(impute_price, axis=1, result_type='expand')
    df['final_price'] = imputed_data[0]
    df['price_imputation_tier'] = imputed_data[1]
    
    imputed_count = (df['price_imputation_tier'] != 'original').sum()
    print(f"      {imputed_count:,} precos imputados")
    
    df['price_imputation_tier'].value_counts()

   Aplicando imputacao de precos...
      3,031 precos imputados


### 6.5 Cálculo de Descontos
**Calculamos a porcentagem de desconto comparando preço original vs. preço final.**  
Criamos também faixas de desconto para facilitar análises segmentadas (0-10%, 10-20%, etc.).


In [28]:
 ##Produtos com descontos
 
df['discount_pct'] = np.where(
    (df['listed_price'].notna()) & (df['listed_price'] > 0) & (df['final_price'].notna()),
    (df['listed_price'] - df['final_price']) / df['listed_price'] * 100,
    0.0
).clip(0, 95)

df['discount_bucket'] = pd.cut(
    df['discount_pct'],
    bins=[-0.1, 0, 10, 20, 30, 50, 95],
    labels=['No Discount', '0-10%', '10-20%', '20-30%', '30-50%', '50%+']
)

df['has_discount'] = df['discount_pct'] > 0

### 6.6 Identificação de Cupons
**Extraímos informações sobre cupons de desconto disponíveis.**  
Marcamos produtos que têm cupom e calculamos a porcentagem de desconto adicional.


In [29]:
df['coupon_discount_pct'] = df['is_couponed'].apply(extract_coupon_percentage)
df['has_coupon'] = (df['coupon_discount_pct'] > 0).astype(bool)
coupon_count = df['has_coupon'].sum()

### 6.7 Faixas de Preço (Price Tiers)

**Categorizamos produtos por faixa de preço para análise de mercado.**  
Isso facilita identificar quais segmentos (Budget, Premium, Luxury) têm melhor desempenho.

In [30]:
print("   Criando faixas de preco...")

def create_price_tier(price):
    """Categoriza produtos por faixa de preco"""
    if pd.isna(price):
        return 'Unknown'
    elif price < 20:
        return 'Budget (< $20)'
    elif price < 50:
        return 'Economy ($20-50)'
    elif price < 100:
        return 'Mid-Range ($50-100)'
    elif price < 200:
        return 'Premium ($100-200)'
    elif price < 500:
        return 'High-End ($200-500)'
    else:
        return 'Luxury ($500+)'

df['price_tier'] = df['final_price'].apply(create_price_tier)

print(f"      Distribuicao por faixa de preco:")
tier_dist = df['price_tier'].value_counts()
for tier, count in tier_dist.items():
    print(f"         - {tier}: {count:,} ({count/len(df)*100:.1f}%)")

   Criando faixas de preco...
      Distribuicao por faixa de preco:
         - Economy ($20-50): 3,963 (24.9%)
         - Premium ($100-200): 3,454 (21.7%)
         - Budget (< $20): 2,972 (18.6%)
         - Mid-Range ($50-100): 2,698 (16.9%)
         - High-End ($200-500): 1,707 (10.7%)
         - Luxury ($500+): 1,144 (7.2%)


### 6.8 Score de Qualidade do Produto
**Combinamos rating e reviews em uma métrica única de 0-100.**  
Útil para criar quadrantes de análise (alta qualidade + baixa venda = oportunidade!).


In [31]:
print("   Calculando quality score...")

def calculate_quality_score(row):
    """
    Combina rating + social proof para score 0-100
    50 pontos: rating normalizado (0-5 -> 0-50)
    50 pontos: reviews em escala log (0-10K+ -> 0-50)
    """
    if pd.isna(row['rating']) or pd.isna(row['number_of_reviews']):
        return None
    
    rating_score = (row['rating'] / 5.0) * 50
    review_score = min(50, (np.log1p(row['number_of_reviews']) / np.log1p(10000)) * 50)
    
    return round(rating_score + review_score, 2)

df['quality_score'] = df.apply(calculate_quality_score, axis=1)

valid_scores = df['quality_score'].notna().sum()
print(f"      {valid_scores:,} produtos com quality score")
print(f"      Score medio: {df['quality_score'].mean():.2f}")
print(f"      Score mediano: {df['quality_score'].median():.2f}")

   Calculando quality score...
      15,557 produtos com quality score
      Score medio: 77.45
      Score mediano: 78.50


### 6.9 Receita Estimada
**Calculamos a receita potencial multiplicando unidades vendidas por preço.**  
Métrica essencial para identificar produtos que geram mais valor financeiro.


In [32]:
df['revenue_last_month'] = df['units_sold_last_month'] * df['final_price']

valid_revenue = df['revenue_last_month'].notna().sum()
total_revenue = df['revenue_last_month'].sum()

### 6.10 Flag: Produto Promovível
**Identificamos produtos prontos para promoção baseado em critérios de qualidade.**  
Rating ≥ 4.0, Reviews ≥ 100, Vendas ≥ 200, Disponível para compra = Produto promovível!


In [33]:
def is_promotable(row):
    """
    Produto promovivel se atende criterios:
    - Rating >= 4.0
    - Reviews >= 100
    - Units sold >= 200
    - Buy box disponivel
    """
    if pd.isna(row['rating']) or pd.isna(row['number_of_reviews']) or pd.isna(row['units_sold_last_month']):
        return False
    
    return (
        row['rating'] >= 4.0 and
        row['number_of_reviews'] >= 100 and
        row['units_sold_last_month'] >= 200 and
        row['buy_box_availability'] == True
    )

df['is_promotable'] = df.apply(is_promotable, axis=1)

promotable_count = df['is_promotable'].sum()

top_promotable = df[df['is_promotable']].groupby('brand').size().sort_values(ascending=False).head(5)

## 7. Selecionando e Renomeando Colunas Finais

**Organizamos o dataset final selecionando apenas colunas relevantes.**  
Renomeamos algumas para nomes mais intuitivos e descartamos colunas temporárias ou redundantes.

In [34]:
df = df.rename(columns={
    'number_of_reviews': 'review_count',
    'current/discounted_price': 'discounted_price',
    'listed_price': 'original_price',
    'bought_in_last_month': 'bought_in_last_month_raw',
    'is_couponed': 'is_couponed_raw',
    'is_best_seller': 'best_seller_badge',
    'is_sponsored': 'sponsored_badge',
    'buy_box_availability': 'available_for_purchase',
    'has_discount': 'is_discounted',
    'has_coupon': 'has_active_coupon',
})

columns_to_keep = [
    'asin',
    'title', 'brand', 'category',
    'rating', 'review_count', 'quality_score',
    'final_price', 'original_price', 'price_tier', 'price_source',
    'discount_pct', 'discount_bucket', 'is_discounted',
    'has_active_coupon', 'coupon_discount_pct',
    'best_seller_badge', 'sponsored_badge', 'available_for_purchase', 'is_promotable',
    'units_sold_last_month', 'revenue_last_month',
    'date', 'hour', 'day_of_week', 'day_name', 'collected_at',
    'bought_in_last_month_raw', 'is_couponed_raw'
]

if 'price_imputation_tier' in df.columns:
    columns_to_keep.append('price_imputation_tier')

columns_to_keep = [col for col in columns_to_keep if col in df.columns]
df_silver = df[columns_to_keep].copy()

print(f"   {len(columns_to_keep)} colunas selecionadas")
print(f"   Shape final: {df_silver.shape}")

   30 colunas selecionadas
   Shape final: (15938, 30)


## 8. Salvando os Dados Processados

In [35]:
print("\nSalvando na camada Silver...")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE_CSV)
df_silver.to_csv(csv_path, index=False, encoding='utf-8')
print(f"   CSV salvo: {csv_path}")
print(f"   Tamanho: {os.path.getsize(csv_path) / 1024**2:.2f} MB")


Salvando na camada Silver...
   CSV salvo: Data Layer/silver/data/data_silver.csv
   Tamanho: 5.60 MB


## 10. Popular Banco de Dados (Silver Layer)

Conecta ao PostgreSQL e popula a tabela `silver.product` com os dados processados.


In [36]:
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()


In [37]:
cur.execute("TRUNCATE TABLE silver.product CASCADE;")
conn.commit()

In [38]:
data = []
for idx, row in df_silver.iterrows():
    # Extrair hora da coleta
    time_value = None
    if pd.notna(row['collected_at']):
        time_value = row['collected_at'].time() if hasattr(row['collected_at'], 'time') else None
    
    data.append((
        idx + 1,  # id
        str(row['asin']) if pd.notna(row['asin']) else None,  # asin
        str(row['title'])[:2000] if pd.notna(row['title']) else None,  # title
        str(row['brand'])[:100] if pd.notna(row['brand']) else 'unknown',  # brand
        str(row['category'])[:50] if pd.notna(row['category']) else 'Other',  # category
        float(row['rating']) if pd.notna(row['rating']) else None,  # rating
        int(row['review_count']) if pd.notna(row['review_count']) else 0,  # total_reviews
        int(row['units_sold_last_month']) if pd.notna(row['units_sold_last_month']) else None,  # purchased_last_month
        float(row['final_price']) if pd.notna(row['final_price']) else None,  # discounted_price
        float(row['original_price']) if pd.notna(row['original_price']) else None,  # original_price
        float(row['discount_pct']) if pd.notna(row['discount_pct']) else 0.0,  # discount_percentage
        bool(row['best_seller_badge']) if pd.notna(row['best_seller_badge']) else False,  # is_best_seller
        bool(row['sponsored_badge']) if pd.notna(row['sponsored_badge']) else False,  # is_sponsored
        bool(row['has_active_coupon']) if pd.notna(row['has_active_coupon']) else False,  # has_coupon
        bool(row['available_for_purchase']) if pd.notna(row['available_for_purchase']) else False,  # buy_box_availability
        float(row['quality_score']) if pd.notna(row['quality_score']) else None,  # quality_score
        str(row['price_tier'])[:30] if pd.notna(row['price_tier']) else 'Unknown',  # price_range
        row['date'].date() if pd.notna(row['date']) else None,  # date
        time_value  # time
    ))

# Inserir em batch
insert_sql = """
    INSERT INTO silver.product 
    (id, asin, title, brand, category, rating, total_reviews, purchased_last_month,
     discounted_price, original_price, discount_percentage, is_best_seller, is_sponsored,
     has_coupon, buy_box_availability, quality_score, price_range, date, time)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

execute_batch(cur, insert_sql, data, page_size=1000)
conn.commit()

print(f"{len(data):,} registros inseridos na silver.product")


15,938 registros inseridos na silver.product
